In [4]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [5]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [6]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [7]:
import os

key = os.getenv("GROQ_API_KEY")
print(key[:12] if key else None)

gsk_hAlS4LZN


In [8]:
llm("so i just discovered a course, can you explain")

'Sure thing! I just need a bit more info to give you the best rundown:\n\n1. **Course title & provider** (e.g., “CS50: Introduction to Computer Science” on edX, “The Complete Digital Marketing Course” on Udemy, etc.).  \n2. **What you’re hoping to get out of it** (e.g., career skill, hobby, foundational knowledge, certification).  \n3. **Any specific parts you’re curious about** (syllabus, assignments, instructor background, cost, duration).\n\nOnce I have those details, I can walk you through the structure, key topics, learning outcomes, and whether it’s a good fit for your goals.'

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
'''question = "I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?"'''
question = "fee details of this course?"

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
answer = llm(prompt)
print(answer)

I don't know.


In [12]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [13]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1407

In [14]:
documents[0]

{'id': '4487db3924',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I access the course modules and materials?',
 'answer': "All course materials are in the GitHub repo. Each module has its own folder (e.g. `01-overview`, `03-mcp`), and cohort-specific homework is under the `cohorts/` folder.\n\nLectures are pre-recorded and available in the YouTube playlist. New workshops or updated videos are announced on Slack and Telegram. If you don't see an announcement, assume everything you need is already there.\n\nEach homework has a strict deadline listed on the schedule; after the deadline the form closes. Submissions appear on the leaderboard. You can earn extra points by sharing your learning publicly with the hashtag #aidevtools and tagging Alexey Grigorev or DataTalksClub."}

In [15]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [25]:
def search(question, course="mlops-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [26]:
search_results = search(question)
print(search_results)

[{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}, {'id': 'c842475338', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Homework: Just found this course, can I still submit homeworks?', 'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "Open/Submittable":